# Load and process the dataset

Dataset: [Fraud Detection — 1M Transactions, 7 Fraud Types](https://www.kaggle.com/datasets/sergionefedov/fraud-detection-1m-transactions-7-fraud-types) (Kaggle).

This cell downloads the dataset, loads it into a `pandas.DataFrame`, and builds a scikit-learn preprocessing pipeline (imputation, scaling, one-hot encoding) so the result is ready to feed straight into a model in the next cell.

**Before running on a fresh machine:** set up a Kaggle API token (`~/.kaggle/kaggle.json`, or `KAGGLE_USERNAME`/`KAGGLE_KEY` as Colab secrets) — see this folder's `README.md`.

In [ ]:
%pip install -q kagglehub scikit-learn pandas numpy

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import kagglehub
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Download the dataset and locate the CSV -------------------------------
dataset_dir = kagglehub.dataset_download(
    "sergionefedov/fraud-detection-1m-transactions-7-fraud-types"
)
csv_candidates = glob.glob(os.path.join(dataset_dir, "**", "*.csv"), recursive=True)
assert csv_candidates, f"No CSV found under {dataset_dir}"
csv_path = csv_candidates[0]
print("Loading:", csv_path)

In [ ]:
# 2. Load and take a first look ---------------------------------------------
df = pd.read_csv(csv_path)
print("shape:", df.shape)
df.info()
df.head()

**Check the output above.** The next cell auto-detects the fraud label and the
column types from `df`'s dtypes/names — if the real column names differ from
what gets picked (print statements below make the guess visible), adjust
`target_col` / `id_like` accordingly before continuing.

In [ ]:
# 3. Identify the fraud label -------------------------------------------------
candidate_targets = [
    c for c in df.columns if any(k in c.lower() for k in ("fraud", "class", "label"))
]
print("Candidate target columns:", candidate_targets)
assert candidate_targets, "Could not auto-detect a fraud/label column — set target_col manually."

# Prefer the low-cardinality column (the binary is_fraud flag) over a
# free-text fraud-type column with 7+ categories.
target_col = min(candidate_targets, key=lambda c: df[c].nunique())
print("Using target column:", target_col)
print(df[target_col].value_counts(normalize=True))

In [ ]:
# 4. Basic cleaning ------------------------------------------------------------
df = df.drop_duplicates()

# Identifier-style columns carry no predictive signal and would just leak
# row identity through one-hot encoding — drop them.
id_like = [
    c for c in df.columns
    if c.lower() in ("transaction_id", "id", "customer_id", "account_id", "card_number")
]
print("Dropping id-like columns:", id_like)

y = df[target_col]
X = df.drop(columns=id_like + [target_col], errors="ignore")

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(numeric_cols)} numeric columns, {len(categorical_cols)} categorical columns")

In [ ]:
# 5. Preprocessing pipeline (scikit-learn) -------------------------------------
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

In [ ]:
# 6. Stratified train/test split + fit -----------------------------------------
# Fraud is a rare class, so stratify to keep the same fraud ratio in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit the preprocessor on train only, then transform test — avoids leaking
# test-set statistics (means/std/categories) into the transform.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed:", X_train_processed.shape)
print("X_test_processed: ", X_test_processed.shape)
print("train fraud rate:", y_train.mean() if y_train.dtype != object else y_train.value_counts(normalize=True))

`X_train_processed`, `X_test_processed`, `y_train`, `y_test` are now ready to
pass into `model.fit(...)` in the next cell. Class imbalance (e.g. SMOTE via
`imbalanced-learn`, or `class_weight="balanced"`) is a modeling-stage choice
left for that step.

## 2. Models [18 pts]

Three different model *classes* trained on the `X_train_processed` /
`y_train` produced above. At least one (Isolation Forest, and arguably
Random Forest too) is from Topic 5 onward, as required.

- **2.1 Model I — Logistic Regression**: a linear, supervised baseline.
  Cheap to train and interpretable (coefficients say which features push a
  transaction toward "fraud"), which makes it a useful reference point
  before reaching for anything fancier.
- **2.2 Model II — Random Forest (Ensemble Methods, Topic 6)**: a bagged
  ensemble of decision trees. Handles non-linear feature interactions the
  logistic regression can't, and is robust to the mixed numeric/one-hot
  feature space without needing extra tuning.
- **2.3 Model III — Isolation Forest (Unsupervised Learning, Topic 5)**:
  fraud detection is a classic anomaly-detection use case — fraud is rare
  and "looks different" from normal transactions, which is exactly the
  assumption Isolation Forest exploits. Unlike the first two models, it is
  fit **without** `y_train` and only uses the labels afterward, to check
  whether its unsupervised notion of "anomalous" lines up with the actual
  fraud labels.

### 2.1 Model I [6 pts]

Implement your first model and fit the dataset you loaded above.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression: a linear classifier, and the standard baseline for
# any binary classification task before trying something more complex.
# `class_weight="balanced"` re-weights the loss inversely to class
# frequency, which matters here because fraud is a small minority of
# transactions — without it, the model could get high accuracy by just
# predicting "not fraud" every time.
model_1 = LogisticRegression(
    max_iter=1000,       # the default (100) often doesn't converge on data this size
    class_weight="balanced",
    random_state=42,
)

# Fit on the preprocessed training data built in the "load and process" step.
model_1.fit(X_train_processed, y_train)
print("Model I (Logistic Regression) trained.")
print("Train accuracy:", model_1.score(X_train_processed, y_train))

### 2.2 Model II [6 pts]

Implement your second model and fit the dataset you loaded above.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest: an ensemble (Topic 6) of many decision trees, each trained
# on a bootstrap sample of the data with a random subset of features
# considered at each split ("bagging"). Averaging many de-correlated trees
# reduces the overfitting a single deep tree would show, and it captures
# non-linear feature interactions that Logistic Regression can't.
model_2 = RandomForestClassifier(
    n_estimators=200,        # number of trees in the ensemble
    max_depth=None,          # let trees grow fully; the ensemble averaging controls variance
    class_weight="balanced", # same rare-class reasoning as Model I
    n_jobs=-1,                # use all available CPU cores — trees train independently
    random_state=42,
)

model_2.fit(X_train_processed, y_train)
print("Model II (Random Forest) trained.")
print("Train accuracy:", model_2.score(X_train_processed, y_train))

### 2.3 Model III [6 pts]

Implement your third model and fit the dataset you loaded above.

In [ ]:
from sklearn.ensemble import IsolationForest

# Isolation Forest: an unsupervised anomaly-detection ensemble (Topic 5).
# It isolates points by recursive random splits — outliers, being "few and
# different", tend to get separated in fewer splits than normal points, so
# the average path length to isolate a point becomes an anomaly score.
#
# `contamination` tells it what fraction of points to ultimately flag as
# anomalies; we set it to the *actual* fraud rate in the training data so
# the flagged fraction is comparable to the real prevalence, rather than
# scikit-learn's default guess of 0.1 (10%).
fraud_rate = y_train.mean() if y_train.dtype != object else (y_train == y_train.mode()[0]).mean()
contamination = min(max(float(fraud_rate), 1e-4), 0.5)  # keep it in a valid, non-degenerate range

model_3 = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42,
    n_jobs=-1,
)

# Note: fit() here does NOT take y_train — this is unsupervised, it only
# ever sees the feature matrix. Labels are used afterward for evaluation.
model_3.fit(X_train_processed)

# IsolationForest.predict returns 1 = "inlier" (normal) and -1 = "outlier"
# (anomaly). Flip that onto the same {0, 1} convention as the fraud label
# so this model's output can be compared/evaluated against y_train.
train_pred_anomaly = (model_3.predict(X_train_processed) == -1).astype(int)
print("Model III (Isolation Forest) trained.")
print("Fraction flagged anomalous on train:", train_pred_anomaly.mean())